# Pre-edit Watermarked Text Generation (T = 1)

This notebook generates 1,000 OPT-1.3B continuations with the Gumbel-max watermark and repeated-context masking. It saves the prompts, generated tokens, pre-edit pivots, and watermark indicators in `pre_edit_T1.zip`.

**Output ZIP contents**

- `dataset.npz`: `tokens_gen`, `S1`, `Ys`, `prompts_tokens`
- `meta.json`: model and watermark settings
- `prompts_text.json`: source C4 documents used to construct prompts


In [ ]:
%pip -q install transformers datasets accelerate sentencepiece


In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUTPUT_NAME = 'pre_edit_T1.zip'
print('Device:', DEVICE)


In [ ]:
def save_pre_edit_zip(output_name: str, meta: dict, arrays: dict, prompt_texts: list[str]) -> Path:
    folder = ROOT / Path(output_name).stem
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

    (folder / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    (folder / 'prompts_text.json').write_text(json.dumps(prompt_texts, indent=2), encoding='utf-8')
    np.savez_compressed(folder / 'dataset.npz', **arrays)

    zip_path = ROOT / output_name
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=folder)
    print('Saved:', zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass
    return zip_path


In [ ]:
_table_generator = torch.Generator(device='cpu')
_table_generator.manual_seed(2971215073)
_TABLE_SIZE = 1_000_003
_FIXED_TABLE = torch.randperm(_TABLE_SIZE, device='cpu', generator=_table_generator)


def hashint(integer_tensor: torch.LongTensor) -> torch.LongTensor:
    return _FIXED_TABLE[integer_tensor.cpu() % _TABLE_SIZE] + 1


def noncomm_prf(input_ids: torch.LongTensor, salt_key: int) -> int:
    key_value = torch.as_tensor(salt_key, dtype=torch.long)
    for entry in input_ids:
        key_value *= hashint(key_value * entry)
        key_value %= 2**32 - 1
    return int(key_value.item())


def seed_rng(generator: torch.Generator, tokens: torch.LongTensor,
             hash_key: int, context_width: int) -> None:
    if tokens.shape[-1] < context_width:
        raise ValueError('The prefix is shorter than the watermark context width.')
    prf_key = noncomm_prf(tokens[0, -context_width:], salt_key=hash_key)
    generator.manual_seed(prf_key)


def gumbel_key(generator: torch.Generator, inputs: torch.LongTensor,
               vocab_size: int, key: int, context_width: int):
    xi_rows = []
    token_order = torch.arange(vocab_size)
    for row in inputs:
        seed_rng(generator, row.unsqueeze(0), key, context_width)
        xi_rows.append(torch.rand((1, vocab_size), generator=generator))
    xi = torch.vstack(xi_rows)
    pi = token_order.unsqueeze(0).repeat(inputs.shape[0], 1)
    return xi, pi


def gumbel_sample(probabilities: torch.Tensor, pi: torch.Tensor, xi: torch.Tensor) -> torch.Tensor:
    selected = torch.argmax(xi ** (1 / torch.gather(probabilities, 1, pi)), dim=1)
    return selected.unsqueeze(-1)


def gumbel_pivot(selected: torch.Tensor, xi: torch.Tensor) -> torch.Tensor:
    return torch.gather(xi, -1, selected.cpu()).squeeze()


In [ ]:
class WatermarkGenerator:
    def __init__(self, model, vocab_size: int, key: int, generation_length: int,
                 temperature: float, context_width: int, non_watermarked_temperature: float):
        self.model = model
        self.vocab_size = vocab_size
        self.key = key
        self.generation_length = generation_length
        self.temperature = temperature
        self.context_width = context_width
        self.non_watermarked_temperature = non_watermarked_temperature

    def repeated_context(self, batch_tokens: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length = batch_tokens.shape
        if sequence_length <= self.context_width:
            return torch.zeros(batch_size, dtype=torch.bool, device=batch_tokens.device)
        current = batch_tokens[:, -self.context_width:].unsqueeze(1)
        history = torch.stack(
            [batch_tokens[:, i:i + self.context_width]
             for i in range(sequence_length - self.context_width)],
            dim=1,
        )
        return (current == history).all(dim=-1).any(dim=1)

    @torch.inference_mode()
    def __call__(self, prompts: torch.Tensor, watermark_probability: float = 1.0):
        batch_size, prompt_length = prompts.shape
        generator = torch.Generator(device='cpu')
        inputs = prompts.to(self.model.device)
        attention_mask = torch.ones_like(inputs)
        past_key_values = None

        watermark_probabilities = torch.full(
            (batch_size,), float(watermark_probability), device=self.model.device
        )
        pivots = np.empty((batch_size, self.generation_length), dtype=np.float32)
        watermark_indicators = np.empty((batch_size, self.generation_length), dtype=np.int8)

        from contextlib import nullcontext
        autocast_context = (
            torch.autocast(device_type='cuda', dtype=torch.bfloat16)
            if self.model.device.type == 'cuda' else nullcontext()
        )

        with autocast_context:
            for position in range(self.generation_length):
                if past_key_values is None:
                    output = self.model(inputs, attention_mask=attention_mask, use_cache=True)
                else:
                    output = self.model(
                        inputs[:, -1:],
                        past_key_values=past_key_values,
                        attention_mask=attention_mask,
                        use_cache=True,
                    )

                logits = output.logits[:, -1]
                probabilities = torch.softmax(logits / self.temperature, dim=-1).cpu()
                ordinary_probabilities = torch.softmax(
                    logits / self.non_watermarked_temperature, dim=-1
                ).cpu()

                xi, pi = gumbel_key(
                    generator, inputs, self.vocab_size, self.key, self.context_width
                )
                history = inputs[:, prompt_length - 1:]
                context_seen = self.repeated_context(history)
                add_watermark = torch.rand((batch_size,), device=self.model.device) < watermark_probabilities
                watermark_active = (~context_seen) & add_watermark

                watermarked_token = gumbel_sample(probabilities, pi, xi).to(self.model.device)
                ordinary_token = torch.multinomial(ordinary_probabilities, 1).to(self.model.device)
                next_token = torch.where(watermark_active[:, None], watermarked_token, ordinary_token)

                pivot = gumbel_pivot(next_token, xi)
                if pivot.ndim == 0:
                    pivot = pivot.unsqueeze(0)
                pivots[:, position] = pivot.cpu().numpy().astype(np.float32)
                watermark_indicators[:, position] = watermark_active.cpu().numpy().astype(np.int8)

                inputs = torch.cat([inputs, next_token], dim=1)
                past_key_values = output.past_key_values
                attention_mask = torch.cat(
                    [attention_mask, attention_mask.new_ones((batch_size, 1))], dim=-1
                )

        return inputs.cpu().numpy().astype(np.int32), pivots, watermark_indicators


In [ ]:
def iter_local_c4(jsonl_path: str):
    with open(jsonl_path, 'r', encoding='utf-8') as handle:
        for line in handle:
            yield json.loads(line)


def build_prompts(tokenizer, num_documents: int, prompt_length: int,
                  generation_length: int, local_path: str):
    if Path(local_path).exists():
        examples = iter_local_c4(local_path)
    else:
        dataset = load_dataset('allenai/c4', 'realnewslike', split='train', streaming=True)
        examples = iter(dataset)

    prompts = []
    source_texts = []
    while len(prompts) < num_documents:
        example = next(examples)
        text = example['text']
        token_ids = tokenizer.encode(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=2028,
        )[0]
        if token_ids.size(0) < prompt_length + generation_length:
            continue
        prompt = token_ids[-(generation_length + prompt_length):-generation_length]
        prompts.append(prompt)
        source_texts.append(text)
    return torch.vstack(prompts), source_texts


## Configuration

The values below reproduce the T = 1 experiment used in the paper.


In [ ]:
MODEL_NAME = 'facebook/opt-1.3b'
NUM_DOCUMENTS = 1000
PROMPT_LENGTH = 50
GENERATION_LENGTH = 400
BATCH_SIZE = 32
TEMPERATURE = 1.0
NON_WATERMARKED_TEMPERATURE = 1.0
CONTEXT_WIDTH = 5
WATERMARK_KEY = 15485863
WATERMARK_PROBABILITY = 1.0
LOCAL_C4_PATH = '/content/TrGoF-main/LLM_codes/c4/c4.json'


## Generate and save the pre-edit dataset


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.manual_seed(WATERMARK_KEY)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
model.config.use_cache = True

vocab_size = int(model.get_output_embeddings().weight.shape[0])
prompts, source_texts = build_prompts(
    tokenizer,
    num_documents=NUM_DOCUMENTS,
    prompt_length=PROMPT_LENGTH,
    generation_length=GENERATION_LENGTH,
    local_path=LOCAL_C4_PATH,
)
prompts_tokens = prompts.cpu().numpy().astype(np.int32)

generator = WatermarkGenerator(
    model=model,
    vocab_size=vocab_size,
    key=WATERMARK_KEY,
    generation_length=GENERATION_LENGTH,
    temperature=TEMPERATURE,
    context_width=CONTEXT_WIDTH,
    non_watermarked_temperature=NON_WATERMARKED_TEMPERATURE,
)

generated_batches = []
pivot_batches = []
indicator_batches = []
for start in range(0, NUM_DOCUMENTS, BATCH_SIZE):
    end = min(NUM_DOCUMENTS, start + BATCH_SIZE)
    full_tokens, pivots, indicators = generator(
        prompts[start:end], watermark_probability=WATERMARK_PROBABILITY
    )
    generated_batches.append(full_tokens[:, PROMPT_LENGTH:].astype(np.int32))
    pivot_batches.append(pivots.astype(np.float32))
    indicator_batches.append(indicators.astype(np.int8))
    print(f'Generated {end}/{NUM_DOCUMENTS} documents')

tokens_gen = np.concatenate(generated_batches, axis=0)
Ys = np.concatenate(pivot_batches, axis=0)
S1 = np.concatenate(indicator_batches, axis=0)

meta = {
    'tag': 'pre_edit_T1',
    'model': MODEL_NAME,
    'watermark_type': 'Gumbel',
    'watermark_probability': WATERMARK_PROBABILITY,
    'T': NUM_DOCUMENTS,
    'batch_size': BATCH_SIZE,
    'prompt_tokens': PROMPT_LENGTH,
    'gen_tokens': GENERATION_LENGTH,
    'temp': TEMPERATURE,
    'non_wm_temp': NON_WATERMARKED_TEMPERATURE,
    'c_window': CONTEXT_WIDTH,
    'key': WATERMARK_KEY,
    'seeding_scheme': 'noncomm_prf',
    'vocab_size_full': vocab_size,
    'device': DEVICE,
    'torch_dtype': 'bfloat16',
}
arrays = {
    'tokens_gen': tokens_gen,
    'S1': S1,
    'Ys': Ys,
    'prompts_tokens': prompts_tokens,
}
save_pre_edit_zip(OUTPUT_NAME, meta, arrays, source_texts)
